# Multi-Agent Banking Customer Support Assistant

A classifier agent routes each customer message to a positive-feedback, negative-feedback or query agent. Tickets and agent logs are stored in SQLite, and a built-in evaluation measures classification accuracy.

**To run:** run the cell below, open the Gradio link, and paste your OpenAI API key in the *API Key* tab. The key is never stored in this notebook.


In [ ]:
# ============================================================
# STEP 1: Install Required Libraries
# ============================================================

!pip install -q gradio langchain langchain-openai pandas


# ============================================================
# STEP 2: Import Required Libraries
# ============================================================

import os
import re
import random
import sqlite3
import pandas as pd
import gradio as gr
from datetime import datetime

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


# ============================================================
# STEP 3: Global Variables
# ============================================================

openai_api_key = None
DB_NAME = "banking_support.db"


# ============================================================
# STEP 4: Create SQLite Database
# ============================================================

def setup_database():
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()

    # Table 1: support tickets
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS support_tickets (
        ticket_number TEXT PRIMARY KEY,
        customer_name TEXT,
        issue_description TEXT,
        status TEXT,
        created_at TEXT
    )
    """)

    # Table 2: logs
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS agent_logs (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        user_message TEXT,
        classification TEXT,
        agent_used TEXT,
        response TEXT,
        created_at TEXT
    )
    """)

    # Add sample ticket data
    cursor.execute("""
    INSERT OR IGNORE INTO support_tickets
    (ticket_number, customer_name, issue_description, status, created_at)
    VALUES
    ('650932', 'Rahul', 'Credit card issue', 'Resolved', ?)
    """, (datetime.now().strftime("%Y-%m-%d %H:%M:%S"),))

    cursor.execute("""
    INSERT OR IGNORE INTO support_tickets
    (ticket_number, customer_name, issue_description, status, created_at)
    VALUES
    ('123456', 'Priya', 'Net banking login issue', 'In Progress', ?)
    """, (datetime.now().strftime("%Y-%m-%d %H:%M:%S"),))

    cursor.execute("""
    INSERT OR IGNORE INTO support_tickets
    (ticket_number, customer_name, issue_description, status, created_at)
    VALUES
    ('987654', 'Amit', 'Debit card replacement issue', 'Open', ?)
    """, (datetime.now().strftime("%Y-%m-%d %H:%M:%S"),))

    conn.commit()
    conn.close()


setup_database()


# ============================================================
# STEP 5: Save OpenAI API Key
# ============================================================

def save_api_key(api_key):
    global openai_api_key

    if api_key is None or api_key.strip() == "":
        return "Please enter your OpenAI API key."

    openai_api_key = api_key.strip()
    os.environ["OPENAI_API_KEY"] = openai_api_key

    return "API key saved successfully."


# ============================================================
# STEP 6: Generate Unique 6-Digit Ticket Number
# ============================================================

def generate_ticket_number():
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()

    while True:
        ticket_number = str(random.randint(100000, 999999))

        cursor.execute(
            "SELECT ticket_number FROM support_tickets WHERE ticket_number = ?",
            (ticket_number,)
        )

        existing_ticket = cursor.fetchone()

        if existing_ticket is None:
            conn.close()
            return ticket_number


# ============================================================
# STEP 7: Log Agent Activity
# ============================================================

def save_log(user_message, classification, agent_used, response):
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()

    cursor.execute("""
    INSERT INTO agent_logs
    (user_message, classification, agent_used, response, created_at)
    VALUES (?, ?, ?, ?, ?)
    """, (
        user_message,
        classification,
        agent_used,
        response,
        datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    ))

    conn.commit()
    conn.close()


# ============================================================
# STEP 8: Classifier Agent
# ============================================================

def classifier_agent(user_message):
    global openai_api_key

    if openai_api_key is None:
        return "API_KEY_MISSING"

    llm = ChatOpenAI(
        model="gpt-4o-mini",
        temperature=0
    )

    prompt = ChatPromptTemplate.from_template("""
You are a banking customer support message classifier.

Classify the user message into exactly one category:

1. Positive Feedback
2. Negative Feedback
3. Query

Rules:
- If the customer is thanking or appreciating, classify as Positive Feedback.
- If the customer is complaining or facing a problem, classify as Negative Feedback.
- If the customer is asking for ticket status or asking a question, classify as Query.

Return only one of these values:
Positive Feedback
Negative Feedback
Query

User Message:
{user_message}
""")

    chain = prompt | llm | StrOutputParser()

    classification = chain.invoke({
        "user_message": user_message
    })

    classification = classification.strip()

    if "Positive" in classification:
        return "Positive Feedback"
    elif "Negative" in classification:
        return "Negative Feedback"
    elif "Query" in classification:
        return "Query"
    else:
        return "Query"


# ============================================================
# STEP 9: Positive Feedback Handler Agent
# ============================================================

def positive_feedback_agent(user_message, customer_name):
    global openai_api_key

    llm = ChatOpenAI(
        model="gpt-4o-mini",
        temperature=0.3
    )

    prompt = ChatPromptTemplate.from_template("""
You are a polite banking customer support assistant.

The customer has given positive feedback.

Create a short, warm, personalized thank-you response.

Customer Name:
{customer_name}

User Message:
{user_message}

Response format:
Thank you for your kind words, [CustomerName]! We’re delighted to assist you.
""")

    chain = prompt | llm | StrOutputParser()

    response = chain.invoke({
        "customer_name": customer_name,
        "user_message": user_message
    })

    return response.strip()


# ============================================================
# STEP 10: Negative Feedback Handler Agent
# ============================================================

def negative_feedback_agent(user_message, customer_name):
    ticket_number = generate_ticket_number()

    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()

    cursor.execute("""
    INSERT INTO support_tickets
    (ticket_number, customer_name, issue_description, status, created_at)
    VALUES (?, ?, ?, ?, ?)
    """, (
        ticket_number,
        customer_name,
        user_message,
        "Unresolved",
        datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    ))

    conn.commit()
    conn.close()

    response = f"""
We apologize for the inconvenience, {customer_name}.
A new ticket #{ticket_number} has been generated, and our support team will follow up shortly.
"""

    return response.strip()


# ============================================================
# STEP 11: Query Handler Agent
# ============================================================

def query_handler_agent(user_message):
    # Extract 6-digit ticket number from message
    ticket_numbers = re.findall(r"\b\d{6}\b", user_message)

    if len(ticket_numbers) == 0:
        return "Please provide a valid 6-digit ticket number to check the status."

    ticket_number = ticket_numbers[0]

    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()

    cursor.execute("""
    SELECT ticket_number, customer_name, issue_description, status, created_at
    FROM support_tickets
    WHERE ticket_number = ?
    """, (ticket_number,))

    ticket = cursor.fetchone()
    conn.close()

    if ticket is None:
        return f"No ticket found with ticket number #{ticket_number}."

    response = f"""
Your ticket #{ticket[0]} is currently marked as: {ticket[3]}.

Customer Name: {ticket[1]}
Issue: {ticket[2]}
Created At: {ticket[4]}
"""

    return response.strip()


# ============================================================
# STEP 12: Main Multi-Agent Coordinator
# ============================================================

def banking_support_ai(customer_name, user_message):
    if openai_api_key is None:
        return "Please enter and save your OpenAI API key first.", "", ""

    if customer_name is None or customer_name.strip() == "":
        customer_name = "Customer"

    if user_message is None or user_message.strip() == "":
        return "Please enter a customer message.", "", ""

    user_message = user_message.strip()
    customer_name = customer_name.strip()

    # Step 1: Classifier Agent
    classification = classifier_agent(user_message)

    if classification == "API_KEY_MISSING":
        return "Please enter and save your OpenAI API key first.", "", ""

    # Step 2: Route to correct agent
    if classification == "Positive Feedback":
        agent_used = "Positive Feedback Handler Agent"
        response = positive_feedback_agent(user_message, customer_name)

    elif classification == "Negative Feedback":
        agent_used = "Negative Feedback Handler Agent"
        response = negative_feedback_agent(user_message, customer_name)

    elif classification == "Query":
        agent_used = "Query Handler Agent"
        response = query_handler_agent(user_message)

    else:
        agent_used = "Fallback Agent"
        response = "Sorry, I could not understand the message clearly."

    # Step 3: Save logs
    save_log(user_message, classification, agent_used, response)

    routing_path = f"""
User Message
↓
Classifier Agent
↓
{classification}
↓
{agent_used}
↓
Final Response
"""

    return response, classification, routing_path


# ============================================================
# STEP 13: View Support Tickets
# ============================================================

def view_support_tickets():
    conn = sqlite3.connect(DB_NAME)

    df = pd.read_sql_query("""
    SELECT
        ticket_number,
        customer_name,
        issue_description,
        status,
        created_at
    FROM support_tickets
    ORDER BY created_at DESC
    """, conn)

    conn.close()

    return df


# ============================================================
# STEP 14: View Agent Logs
# ============================================================

def view_agent_logs():
    conn = sqlite3.connect(DB_NAME)

    df = pd.read_sql_query("""
    SELECT
        id,
        user_message,
        classification,
        agent_used,
        response,
        created_at
    FROM agent_logs
    ORDER BY id DESC
    """, conn)

    conn.close()

    return df


# ============================================================
# STEP 15: Simple Evaluation Function
# ============================================================

def run_simple_evaluation():
    test_cases = [
        {
            "message": "Thanks for resolving my credit card issue.",
            "expected": "Positive Feedback"
        },
        {
            "message": "My debit card replacement still has not arrived.",
            "expected": "Negative Feedback"
        },
        {
            "message": "Could you check the status of ticket 650932?",
            "expected": "Query"
        },
        {
            "message": "I am unhappy with the loan processing delay.",
            "expected": "Negative Feedback"
        },
        {
            "message": "Great service from your bank team.",
            "expected": "Positive Feedback"
        }
    ]

    results = []

    correct_count = 0

    for case in test_cases:
        predicted = classifier_agent(case["message"])

        if predicted == case["expected"]:
            result = "Pass"
            correct_count += 1
        else:
            result = "Fail"

        results.append({
            "Message": case["message"],
            "Expected": case["expected"],
            "Predicted": predicted,
            "Result": result
        })

    accuracy = (correct_count / len(test_cases)) * 100

    result_df = pd.DataFrame(results)

    summary = f"Classification Accuracy: {accuracy}%"

    return summary, result_df


# ============================================================
# STEP 16: Create Gradio UI
# ============================================================

with gr.Blocks(title="Banking Customer Support AI Agent") as app:

    gr.Markdown("""
    # Banking Customer Support AI Agent

    This is a simple multi-agent GenAI system for banking customer support.

    It has:
    - Classifier Agent
    - Positive Feedback Handler Agent
    - Negative Feedback Handler Agent
    - Query Handler Agent
    - SQLite Ticket Database
    - Logs and Evaluation
    """)

    # --------------------------------------------------------
    # Tab 1: API Key
    # --------------------------------------------------------

    with gr.Tab("1. API Key"):
        gr.Markdown("### Enter your OpenAI API Key")

        api_key_input = gr.Textbox(
            label="OpenAI API Key",
            type="password",
            placeholder="Paste your OpenAI API key here"
        )

        api_key_button = gr.Button("Save API Key")

        api_key_status = gr.Textbox(
            label="API Key Status"
        )

        api_key_button.click(
            fn=save_api_key,
            inputs=api_key_input,
            outputs=api_key_status
        )

    # --------------------------------------------------------
    # Tab 2: Customer Support Agent
    # --------------------------------------------------------

    with gr.Tab("2. Customer Support Agent"):
        gr.Markdown("### Enter a banking customer message")

        customer_name_input = gr.Textbox(
            label="Customer Name",
            placeholder="Example: Rahul"
        )

        user_message_input = gr.Textbox(
            label="Customer Message",
            lines=4,
            placeholder="Example: My debit card replacement still has not arrived."
        )

        submit_button = gr.Button("Run Multi-Agent Support AI")

        final_response_output = gr.Textbox(
            label="Final Response",
            lines=6
        )

        classification_output = gr.Textbox(
            label="Classification"
        )

        routing_output = gr.Textbox(
            label="Agent Routing Path",
            lines=8
        )

        submit_button.click(
            fn=banking_support_ai,
            inputs=[customer_name_input, user_message_input],
            outputs=[final_response_output, classification_output, routing_output]
        )

    # --------------------------------------------------------
    # Tab 3: Support Tickets
    # --------------------------------------------------------

    with gr.Tab("3. Support Tickets"):
        gr.Markdown("### View all support tickets")

        tickets_button = gr.Button("Refresh Tickets")

        tickets_table = gr.Dataframe(
            label="Support Tickets"
        )

        tickets_button.click(
            fn=view_support_tickets,
            inputs=[],
            outputs=tickets_table
        )

    # --------------------------------------------------------
    # Tab 4: Logs
    # --------------------------------------------------------

    with gr.Tab("4. Logs"):
        gr.Markdown("### View agent logs")

        logs_button = gr.Button("Refresh Logs")

        logs_table = gr.Dataframe(
            label="Agent Logs"
        )

        logs_button.click(
            fn=view_agent_logs,
            inputs=[],
            outputs=logs_table
        )

    # --------------------------------------------------------
    # Tab 5: Evaluation
    # --------------------------------------------------------

    with gr.Tab("5. Evaluation"):
        gr.Markdown("### Test classification accuracy")

        evaluation_button = gr.Button("Run Evaluation")

        evaluation_summary = gr.Textbox(
            label="Evaluation Summary"
        )

        evaluation_table = gr.Dataframe(
            label="Evaluation Results"
        )

        evaluation_button.click(
            fn=run_simple_evaluation,
            inputs=[],
            outputs=[evaluation_summary, evaluation_table]
        )


# ============================================================
# STEP 17: Launch the App
# ============================================================

app.launch(share=True)